In [79]:

import os
import time
import logging

from dataclasses import dataclass, field
from typing import Optional

import requests
import pandas as pd

from dotenv import load_dotenv
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification

# If your .env is in a parent directory, point load_dotenv at it explicitly
load_dotenv(os.path.join(os.getcwd(), "..", ".env"))

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)-8s  %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger(__name__)

# paste the rest of your class definitions here (everything except __main__)


In [87]:
import sys


# Add the folder containing finbert_sentiment.py to the path
sys.path.insert(0, os.path.join(os.getcwd(), ".."))  # adjust if needed

from src.sentiment import (
    SentimentAnalyzer,
    FinBERTScorer,
    NewsFetcher,
    compute_daily_sentiment,
    compute_weekly_sentiment,
    compute_quarterly_sentiment,
    print_summary,
    GDP_RELEVANT_TOPICS,
    AlphaVantageRateLimitError,
)

from src.sentiment_visuals import plot_all

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), "..", ".env"))



True

In [ ]:
# Base directory
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

# Data folders
RAW_DATA_DIR       = os.path.join(BASE_DIR, "data", "raw")
PROCESSED_DATA_DIR = os.path.join(BASE_DIR, "data", "processed")

# Output folders
FIGURES_DIR        = os.path.join(BASE_DIR, "outputs", "figures")
TABLES_DIR         = os.path.join(BASE_DIR, "outputs", "tables")

In [104]:

fetcher = NewsFetcher(api_key=os.getenv("ALPHA_VANTAGE_API_KEY"))
scorer  = FinBERTScorer(device=-1)

windows = [
    ("20260101T0000", "20260131T2359"),
    ("20260201T0000", "20260228T2359"),
    ("20260301T0000", "20260331T2359"),
    ("20260401T0000", "20260426T2359"),
]

all_dfs = []
for time_from, time_to in windows:
    try:
        articles = fetcher.fetch(topics=None, limit=50, sort="EARLIEST", time_from=time_from)
        if not articles:
            print(f"{time_from[:6]}: 0 articles — skipping")
            continue

        texts  = [a.title for a in articles]
        scores = scorer.score(texts)

        rows = []
        for a, s in zip(articles, scores):
            rows.append({
                "published_at":    a.published_at,
                "source":          a.source,
                "title":           a.title,
                "url":             a.url,
                "topics":          ", ".join(a.topics),
                "finbert_label":   s["label"],
                "finbert_score":   round(s["score"], 4),
                "sentiment_value": {"positive": 1.0, "neutral": 0.0, "negative": -1.0}[s["label"]],
            })

        batch_df = pd.DataFrame(rows)
        batch_df["published_at"] = pd.to_datetime(
            batch_df["published_at"], format="%Y%m%dT%H%M%S", errors="coerce"
        )
        all_dfs.append(batch_df)
        print(f"{time_from[:6]}: {len(batch_df)} articles  "
              f"({batch_df['published_at'].min().date()} → {batch_df['published_at'].max().date()})")

    except AlphaVantageRateLimitError:
        print(f"{time_from[:6]}: rate limit hit — stopping")
        break

df = pd.concat(all_dfs, ignore_index=True).drop_duplicates(subset=["url"])
print(f"\nTotal unique articles : {len(df)}")
print(f"Date range            : {df['published_at'].min()} → {df['published_at'].max()}")

# Save
df.to_csv(os.path.join(RAW_DATA_DIR, "articles_raw_batched.csv"), index=False)

14:13:45  INFO      Loading ProsusAI/finbert …
14:13:46  INFO      Model loaded.
14:13:46  INFO      Fetching up to 50 articles  topics=(none) …


202601: rate limit hit — stopping


ValueError: No objects to concatenate

In [98]:

df = pd.read_csv(os.path.join(RAW_DATA_DIR, "articles_raw_batched.csv"))
df["published_at"] = pd.to_datetime(df["published_at"])

daily   = compute_daily_sentiment(df)
weekly  = compute_weekly_sentiment(df)
print_summary(df, weekly)

# Compute net sentiment directly from the batched df instead of result
net_sentiment = (
    (df["finbert_label"] == "positive").sum() - 
    (df["finbert_label"] == "negative").sum()
) / len(df)

print("Net sentiment:", round(net_sentiment, 4))



  FinBERT Sentiment Summary
  Articles analysed : 50
  Positive          : 32.0%
  Neutral           : 48.0%
  Negative          : 20.0%
  Net sentiment     : +0.120  (range −1 to +1)

Top 5 most positive headlines:
  [0.96] Teledyne (TDY) Posts Healthy Earnings As Focus Turns To Space
  [0.95] SpaceX IPO Could Boost ATI Inc (ATI) Prospects
  [0.94] EQT Corporation price target raised to $77 from $76 at Jefferies
  [0.94] Cisco Systems, Inc. stock (US17275R1023): Is its AI networking edge strong enough to unloc
  [0.94] Dollar Tree Signs Multi-Year Agreement With Jimmie Johnson, Expands Relationship With Lega

Top 5 most negative headlines:
  [0.97] Blackstone-affiliated UKG cuts 950 jobs
  [0.96] Bank of America Cuts Pentair (PNR) PT to $88, Sees Weak Q1 Ahead
  [0.95] APA Corp Stock Declines 2.9% Amid Crude Oil Price Drop on April 26, 2026 - News and Statis
  [0.91] BMO Trims IBM Target on Soft Growth, Wedbush Sees AI Tailwinds
  [0.90] Tractor Supply, Lululemon, and Northrop Grumma

In [100]:

# Rebuild the series from the batched df
df_copy = df.copy()
df_copy["period"] = df_copy["published_at"].dt.to_period("Q")

def net_sentiment_agg(g):
    total = len(g)
    pos   = (g["finbert_label"] == "positive").sum()
    neg   = (g["finbert_label"] == "negative").sum()
    return round((pos - neg) / total, 4)

series = df_copy.groupby("period").apply(net_sentiment_agg).rename("net_sentiment")

plot_all(
    df=df,
    series=series,
    output_dir=os.path.join(BASE_DIR, "outputs", "figures", "sentiment_plots"),
)


Saved: d:\College\Wayne State\Graduate\Winter 2026\IE 7860\group4-gdp\outputs\figures\sentiment_plots\plot1_sentiment_pie.png
Saved: d:\College\Wayne State\Graduate\Winter 2026\IE 7860\group4-gdp\outputs\figures\sentiment_plots\plot2_sentiment_over_time.png
Saved: d:\College\Wayne State\Graduate\Winter 2026\IE 7860\group4-gdp\outputs\figures\sentiment_plots\plot3_source_sentiment.png
Saved: d:\College\Wayne State\Graduate\Winter 2026\IE 7860\group4-gdp\outputs\figures\sentiment_plots\plot4_quarterly_sentiment.png

All plots saved to: d:\College\Wayne State\Graduate\Winter 2026\IE 7860\group4-gdp\outputs\figures\sentiment_plots


In [101]:

print(f"Articles pulled  : {len(df)}")
print(f"Earliest article : {df['published_at'].min()}")
print(f"Latest article   : {df['published_at'].max()}")
print(f"Date range       : {(df['published_at'].max() - df['published_at'].min()).days} days")

Articles pulled  : 50
Earliest article : 2026-04-26 15:39:01
Latest article   : 2026-04-26 17:40:03
Date range       : 0 days
